In [ ]:
# to run this notebook from terminal
# jupyter nbconvert --execute --to notebook --inplace scripts/stop_data_processing.ipynb

In [1]:
suppressMessages(library(tidytable))
suppressMessages(library(tidyverse))

In [2]:
start_date = ymd("2021-03-01")
end_date = ymd("2026-04-30")

In [3]:
df_new = read_csv("../data/raw/stop_data_2024_current.csv")
# df_new

New names:
• `` -> `...58`


Warning message:
"One or more parsing issues, call `problems()` on your data frame for details, e.g.:
  dat <- vroom(...)
  problems(dat)"


Rows: 10931 Columns: 58


── Column specification ───────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr  (40): LEA_Record_ID, StopDate, CallForService, WelfareWellnessCheck, In...
dbl   (9): PersonNum, StopDuration, Latitude, Longitude, BlockNumber, Highwa...
lgl   (8): OtherLocationDesc, IsK12School, SchoolName, StopForAStudent, Traf...
time  (1): StopTime



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [4]:
df_old = read_csv("../data/raw/stop_data_2020_2023.csv")
# df_old |>
#     arrange(LEA_RecordID) |>
#     head(n = 14) |>
#     select(LEA_RecordID, DateTimeOfStop, ActionsTaken)

Warning message:
"One or more parsing issues, call `problems()` on your data frame for details, e.g.:
  dat <- vroom(...)
  problems(dat)"


Rows: 18090 Columns: 44


── Column specification ───────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (37): LEA_RecordID, DateTimeOfStop, RespToServiceCall, InfoBasedStop, Ty...
dbl  (6): PersonNum, DateTimeSortable, DurationOfStop, Latitude, Longitude, Age
lgl  (1): GenderNonconforming



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
df_new |>
#   mutate(StopDate = mdy(StopDate)) |>
#   summarise(min_date = min(StopDate), max_date = max(StopDate))
  distinct(TrafficViolationType)
  # filter(!(is.na(VerbalWarningOffense))) |> select(VerbalWarningOffense)
#   # filter(is.na(Longitude)) # because these are so few we could just exclude from the analysis for ease of analysis

TrafficViolationType
<chr>
Moving violation
NA
Equipment violation
Non-moving violation including registration


In [6]:
split_column = function(df, col){
    df |>
        filter(.data[[col]] != "None") |>
        # distinct(.data[[col]])
        pull(.data[[col]]) |>
        str_split("\\|") |>
        unlist() |>
        str_trim() |>
        unique()
}

In [7]:
new_nonforce = split_column(df_new, "NonForceActionsTaken")
new_force = split_column(df_new, "ForceActionsTaken")
old = split_column(df_old, "ActionsTaken")
old_unique = setdiff(old, c(new_nonforce, new_force))

force_actions = c(new_force, old_unique[(!str_detect(old_unique, "search"))])
nonforce_actions = c(new_nonforce, old_unique[str_detect(old_unique, "search")])

df_old_actions = df_old |>
    separate_longer_delim(ActionsTaken, delim = "\\|") |>
    mutate(ActionsTaken = str_trim(ActionsTaken)) |>
    mutate(
        type = case_when(
            ActionsTaken %in% force_actions ~ "force",
            ActionsTaken %in% nonforce_actions ~ "nonforce",
        )
    ) |>
    group_by(LEA_RecordID) |>
    summarise(
        ForceActionsTaken = str_c(ActionsTaken[type == "force"], collapse = "|"),
        NonForceActionsTaken = str_c(ActionsTaken[type == "nonforce"], collapse = "|")
    ) |>
    mutate(
        NonForceActionsTaken = ifelse(NonForceActionsTaken %in% c("", NA), "None", NonForceActionsTaken),
        ForceActionsTaken = ifelse(ForceActionsTaken %in% c("", NA), "None", ForceActionsTaken),
        ) |>
    select(LEA_RecordID, NonForceActionsTaken, ForceActionsTaken)


In [8]:
df_old |>
#   mutate(DateTimeOfStop = mdy_hm(DateTimeOfStop)) |>
#   summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))
  # filter(is.na(Longitude)) # none
  distinct(TrafficViolationType)

names(df_old)

# union(split_column(df_old, "ResultOfStop"), split_column(df_new, "ResultOfStop"))
# split_column(df_old, "ResultOfStop")
# split_column(df_new, "ResultOfStop")

TrafficViolationType
<chr>
NA
Equipment violation
Moving violation
Non-moving violation including registration


[1] "LEA_RecordID"             "PersonNum"               
 [3] "DateTimeOfStop"           "DateTimeSortable"        
 [5] "DurationOfStop"           "RespToServiceCall"       
 [7] "InfoBasedStop"            "TypeOfStop"              
 [9] "LocationDescription"      "City"                    
[11] "Latitude"                 "Longitude"               
[13] "IsK12School"              "SchoolName"              
[15] "StopForAStudent"          "RacePerceivedPriorToStop"
[17] "EthnicitySet"             "Ethnicity"               
[19] "Gender"                   "GenderNonconforming"     
[21] "LGBT"                     "Age"                     
[23] "LimitedEnglish"           "DisabilitySet"           
[25] "Disability"               "CityOfResidence"         
[27] "PrimaryReasonForStop"     "TrafficViolationType"    
[29] "TrafficViolation"         "BasisOfSuspicion"        
[31] "SuspectedViolation"       "ActionsTaken"            
[33] "ConsentToSearchPerson"    "ConsentToSearchProperty" 
[35] "BasisForSearch"           "BasisForPropertySeizure" 
[37] "PropertySeized"           "ContrabandOrEvidence"    
[39] "WeaponsFound"             "ResultOfStop"            
[41] "WarningOffense"           "CitationForInfraction"   
[43] "InFieldCiteAndRelease"    "CustodialArrestNoWarrant"

In [9]:
df_old_cleaned = df_old |>
  mutate(
    DateTimeOfStop = mdy_hm(DateTimeOfStop, tz = "America/Los_Angeles"),
    RespToServiceCall = ifelse(RespToServiceCall == "Yes", "Y", "N"),
    TypeOfStop = ifelse(
      TypeOfStop == "Vehicle", "Vehicular Stop", str_c(TypeOfStop, " Stop")
    ),
    LocationType = "LatLonPt",
  ) |>
  filter(
    # old data ends around mid-2024, exclude the 2024 data
    DateTimeOfStop >= start_date, year(DateTimeOfStop) <= 2023,
    City == "Berkeley",
  ) |>
  left_join(df_old_actions, by = "LEA_RecordID") |>
  select(LEA_Record_ID = LEA_RecordID, PersonNum, DateTimeOfStop,
    CallForService = RespToServiceCall, TypeOfStopDesc = TypeOfStop, StopDuration = DurationOfStop,
    City, LocationType, Latitude, Longitude, LocationDescription,
    TrafficViolationType, NonForceActionsTaken, ForceActionsTaken, PropertySeized, WeaponsFound, ContrabandOrEvidence, ResultOfStop,
    WarningOffense, CitationForInfractionOffense = CitationForInfraction,
    InfieldCiteRelOffense = InFieldCiteAndRelease, CustodialArrestNoWarrantOffense = CustodialArrestNoWarrant,
  )

df_old_cleaned |>
  summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))

# head(df_old_cleaned, 5)

min_date,max_date
<dttm>,<dttm>
2021-03-01 00:03:00,2023-12-31 20:06:00


In [10]:
df_new_cleaned = df_new |>
  mutate(
    DateTimeOfStop = mdy_hms(str_c(StopDate, " ", StopTime), tz = "America/Los_Angeles"),
    City = str_to_title(City),
    WarningOffense = case_when(
      is.na(VerbalWarningOffense) & is.na(WrittenWarningOffense) ~ NA,
      is.na(VerbalWarningOffense) & !(is.na(WrittenWarningOffense)) ~ WrittenWarningOffense,
      !(is.na(VerbalWarningOffense)) & is.na(WrittenWarningOffense) ~ VerbalWarningOffense,
      !(is.na(VerbalWarningOffense)) & !(is.na(WrittenWarningOffense)) ~ str_c(VerbalWarningOffense, "|", WrittenWarningOffense),
    )
  ) |>
  filter(
    DateTimeOfStop <= end_date, # new data starts at 2024
    City == "Berkeley"
  ) |> 
  select(LEA_Record_ID, PersonNum, DateTimeOfStop,
    CallForService, TypeOfStopDesc, StopDuration,
    City, LocationType, Latitude, Longitude,
    BlockNumber, StreetName, CrossStreet1, CrossStreet2, Highway, ClosestExit, OtherLocationDesc, # to be updated once i know how location data is processed
    TrafficViolationType, NonForceActionsTaken, ForceActionsTaken, PropertySeized, WeaponsFound, ContrabandOrEvidence, ResultOfStop,
    WarningOffense, CitationForInfractionOffense, InfieldCiteRelOffense, CustodialArrestNoWarrantOffense
  )

# df_new_cleaned |>
#   select(WarningOffense)
#   summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))


In [11]:
df = full_join(df_old_cleaned, df_new_cleaned) |>
    mutate(DateTimeOfStop = format(DateTimeOfStop, "%Y-%m-%d %H:%M:%S%z"))

write_csv(df, "../data/clean/stop_data.csv")

Joining with `by = join_by(LEA_Record_ID, PersonNum, DateTimeOfStop, CallForService, TypeOfStopDesc,
StopDuration, City, LocationType, Latitude, Longitude, TrafficViolationType, NonForceActionsTaken,
ForceActionsTaken, PropertySeized, WeaponsFound, ContrabandOrEvidence, ResultOfStop, WarningOffense,
CitationForInfractionOffense, InfieldCiteRelOffense, CustodialArrestNoWarrantOffense)`


In [12]:
test = read_csv("../data/clean/stop_data.csv")
test # |> split_column("PropertySeized")

Warning message:
"One or more parsing issues, call `problems()` on your data frame for details, e.g.:
  dat <- vroom(...)
  problems(dat)"


Rows: 24733 Columns: 29


── Column specification ───────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr  (18): LEA_Record_ID, CallForService, TypeOfStopDesc, City, LocationType...
dbl   (5): PersonNum, StopDuration, Latitude, Longitude, BlockNumber
lgl   (5): CrossStreet1, CrossStreet2, Highway, ClosestExit, OtherLocationDesc
dttm  (1): DateTimeOfStop



ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


LEA_Record_ID,PersonNum,DateTimeOfStop,CallForService,TypeOfStopDesc,StopDuration,City,LocationType,Latitude,Longitude,⋯,CitationForInfractionOffense,InfieldCiteRelOffense,CustodialArrestNoWarrantOffense,BlockNumber,StreetName,CrossStreet1,CrossStreet2,Highway,ClosestExit,OtherLocationDesc
<chr>,<dbl>,<dttm>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,⋯,<chr>,<chr>,<chr>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
BPD000000641,1,2021-03-01 08:03:00,N,Vehicular Stop,3,Berkeley,LatLonPt,37.868,-122.258,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
BPD000000646,1,2021-03-01 08:09:00,N,Pedestrian Stop,20,Berkeley,LatLonPt,37.869,-122.272,⋯,NA,NA,PC 417(A)(2) EXHIBIT FIREARM (M),NA,NA,NA,NA,NA,NA,NA
BPD000000642,1,2021-03-01 08:37:00,N,Bicycle Stop,5,Berkeley,LatLonPt,37.866,-122.301,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
BPD000000643,1,2021-03-01 08:45:00,Y,Pedestrian Stop,25,Berkeley,LatLonPt,37.875,-122.270,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
BPD000000649,1,2021-03-01 13:38:00,Y,Vehicular Stop,2,Berkeley,LatLonPt,37.871,-122.280,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
BPD000000650,1,2021-03-01 19:49:00,N,Vehicular Stop,7,Berkeley,LatLonPt,37.855,-122.239,⋯,VC 22350 UNSAFE SPEED:PREVAIL COND (I),NA,NA,NA,NA,NA,NA,NA,NA,NA
BPD000000651,1,2021-03-01 20:11:00,N,Vehicular Stop,8,Berkeley,LatLonPt,37.855,-122.238,⋯,VC 22350 UNSAFE SPEED:PREVAIL COND (I),NA,NA,NA,NA,NA,NA,NA,NA,NA
BPD000000652,1,2021-03-01 20:24:00,N,Vehicular Stop,8,Berkeley,LatLonPt,37.858,-122.242,⋯,VC 22350 UNSAFE SPEED:PREVAIL COND (I),NA,NA,NA,NA,NA,NA,NA,NA,NA
BPD000000654,1,2021-03-01 21:00:00,N,Vehicular Stop,15,Berkeley,LatLonPt,37.869,-122.273,⋯,VC 21950(A) FAIL YIELD TO PED:XWALKS (I),NA,NA,NA,NA,NA,NA,NA,NA,NA
